In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    DINOv3ViTModel,
    UperNetForSemanticSegmentation, 
    DINOv3ViTConfig,
    UperNetConfig,
    UperNetForSemanticSegmentation,
    ResNetConfig,
    AutoBackbone,
    BeitForSemanticSegmentation,
    BeitConfig,
)
from transformers.models.beit.modeling_beit import BeitFCNHead, BeitFPNNeck, BeitUperHead
from transformers.configuration_utils import PreTrainedConfig, SubConfigSpec
from transformers.backbone_utils import load_backbone
from transformers.modeling_utils import PreTrainedModel
from transformers.modeling_outputs import SemanticSegmenterOutput

from types import SimpleNamespace

# Model Decision Notes:

Dinov3
| size | s | b | l |
|------|---|---|---|
| hidden_size | 384 | 768 | 1024 |
| layers | 12 | 12 | 24 | 



Exploration Notes:
1. Directly initializing dinov3 with upernet does not create feature pyramids
2. We follow Beit architecture and use their FPN neck and upernet head

In [ ]:
# a = DINOv3ViTModel.from_pretrained("facebook/dinov3-vits16-pretrain-lvd1689m")
# a = DINOv3ViTModel.from_pretrained("facebook/dinov3-vitb16-pretrain-lvd1689m")
# a = DINOv3ViTModel.from_pretrained("facebook/dinov3-vitl16-pretrain-lvd1689m")

In [ ]:
# upernet_config = UperNetConfig(
#     backbone_config=resnet_config,
#     hidden_size=512,
#     pool_scales=(1, 2, 3, 6),
#     num_labels=19,
#     use_auxiliary_head=False,
# )

# model = UperNetForSemanticSegmentation(upernet_config)
# model.to("cuda", torch.bfloat16)
# model.eval()

In [ ]:
beit_config = BeitConfig.from_pretrained("microsoft/beit-base-patch16-224", out_indices=[3, 5, 7, 11])
model = BeitForSemanticSegmentation(beit_config)
model.to("cuda", torch.bfloat16)
model.eval()
img = torch.randn(1, 3, 256, 512)
img = img.to("cuda", torch.bfloat16)

out = model(img)
out.logits.shape




In [ ]:
# beit uper head needs pool_scales, hidden_size, num_labels 
# beit fpn neck need hidden_size
# beit fcn head needs hidden_size, auxiliary_channels, auxiliary_num_convs, auxiliary_concat_input

In [ ]:
# My implementation of config
class DINOv3ForSemanticSegmentationConfig(PreTrainedConfig):
    model_type = "beit_dinov3"

    sub_configs_defaults = {
        "dinov3_config": SubConfigSpec(
            config_class=DINOv3ViTConfig,
            model_type="dinov3_vit",
            init_kwargs={
                "out_indices": (4, 6, 8, 12),
            },
        ),
    }
    
    dinov3_config: DINOv3ViTConfig | None = None
    # Hub id for backbone weights. load_backbone only builds the module from dinov3_config.
    backbone: str | None = None

    hidden_size: int = 512
    initializer_range: float = 0.02
    
    # upernet head
    pool_scales: tuple[int, ...] = (1, 2, 3, 6)
    num_labels: int = 19

    # auxiliary head
    use_auxiliary_head: bool = True
    auxiliary_loss_weight: float = 0.4
    auxiliary_in_channels: int | None = None
    auxiliary_channels: int = 256
    auxiliary_num_convs: int = 1
    auxiliary_concat_input: bool = False

    semantic_loss_ignore_index: int = 255


In [ ]:
dinvo3_config = DINOv3ViTConfig.from_pretrained("facebook/dinov3-vitb16-pretrain-lvd1689m", out_indices=(4, 6, 8, 12))

In [ ]:
def bn_to_gn(module: nn.Module, groups: int = 32) -> nn.Module:
    """Swap BatchNorm2d for GroupNorm: training batches mix labeled and unlabeled frames."""
    for name, child in module.named_children():
        if isinstance(child, nn.BatchNorm2d):
            setattr(module, name, nn.GroupNorm(groups, child.num_features))
        else:
            bn_to_gn(child, groups)
    return module



In [ ]:
class DINOv3ForSemanticSegmentation(PreTrainedModel):
    def __init__(self, config: DINOv3ForSemanticSegmentationConfig) -> None:
        super().__init__()

        self.backbone = load_backbone(config.dinov3_config)

        if len(self.config.out_indices) != 4:
            raise ValueError(
                "DINOv3ForSemanticSegmentationConfig requires config.out_indices to be a list of 4 integers, "
                "specifying which features to use from the backbone. One can use [3, 5, 7, 11] in case of "
                "a base-sized architecture."
            )
        self.fpn = BeitFPNNeck(config)

        # Semantic segmentation head(s)
        self.decode_head = BeitUperHead(config)
        self.auxiliary_head = BeitFCNHead(config) if config.use_auxiliary_head else None

        # Initialize weights and apply final processing
        self.post_init()

    def forward(
        self,
        pixel_values: torch.Tensor | None = None,
        labels: torch.Tensor | None = None,
        return_dict: bool | None = None,
        **kwargs,
    ) -> SemanticSegmenterOutput:
        _, _, height, width = pixel_values.shape

        return_dict = return_dict if return_dict is not None else self.config.return_dict

        feature_maps = self.backbone(pixel_values).feature_maps
        feature_maps = self.fpn(feature_maps)

        logits = self.decode_head(feature_maps)
        logits = F.interpolate(logits, size=(height, width), mode="bilinear", align_corners=False)


        auxiliary_logits = None
        if self.auxiliary_head is not None: # and self.training:
            auxiliary_logits = self.auxiliary_head(feature_maps)
            auxiliary_logits = F.interpolate(auxiliary_logits, size=(height, width), mode="bilinear", align_corners=False)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=self.config.semantic_loss_ignore_index)
            loss = loss_fct(logits, labels)
            if auxiliary_logits is not None and self.config.use_auxiliary_head:
                loss = loss + self.config.auxiliary_loss_weight * loss_fct(auxiliary_logits, labels)

        # TODO: check if we need to return hidden_states and attentions
        if not return_dict:
            return (loss, logits) if loss is not None else logits

        return SemanticSegmenterOutput(
            loss=loss,
            logits=logits,
        )


In [ ]:

class DINOv3ForSemanticSegmentation(nn.Module):
    def __init__(self, config: DINOv3ForSemanticSegmentationConfig) -> None:
        super().__init__()
        self.config = config

        # load_backbone reads `backbone_config` and returns DINOv3ViTBackbone.
        # It does not download hub weights, so copy them in when a checkpoint is set.
        self.backbone = load_backbone(SimpleNamespace(backbone_config=config.dinov3_config))
        if config.backbone:
            pretrained = AutoBackbone.from_pretrained(config.backbone, config=config.dinov3_config)
            self.backbone.load_state_dict(pretrained.state_dict())
            del pretrained
        self.backbone.requires_grad_(False).eval()

        # BEiT neck and heads use one width. Project DINOv3 maps onto config.hidden_size first.
        head_cfg = BeitConfig(
            hidden_size=config.hidden_size,
            num_labels=config.num_labels,
            pool_scales=config.pool_scales,
            auxiliary_channels=config.auxiliary_channels,
            auxiliary_num_convs=config.auxiliary_num_convs,
            auxiliary_concat_input=config.auxiliary_concat_input,
        )
        self.projections = nn.ModuleList(
            nn.Conv2d(channels, config.hidden_size, kernel_size=1) for channels in self.backbone.channels
        )
        self.fpn = bn_to_gn(BeitFPNNeck(head_cfg))
        self.decode_head = bn_to_gn(BeitUperHead(head_cfg))
        self.auxiliary_head = bn_to_gn(BeitFCNHead(head_cfg)) if config.use_auxiliary_head else None

    def train(self, mode: bool = True):
        super().train(mode)
        self.backbone.eval()
        return self

    def forward(self, pixel_values: torch.Tensor, labels: torch.Tensor | None = None) -> SemanticSegmenterOutput:
        _, _, height, width = pixel_values.shape

        feature_maps = self.backbone(pixel_values).feature_maps
        feature_maps = tuple(proj(fmap) for proj, fmap in zip(self.projections, feature_maps))
        feature_maps = self.fpn(feature_maps)

        def up(t):
            return F.interpolate(t, size=(height, width), mode="bilinear", align_corners=False)

        logits = up(self.decode_head(feature_maps))
        auxiliary_logits = None
        if self.auxiliary_head is not None and self.training:
            auxiliary_logits = up(self.auxiliary_head(feature_maps))

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels, ignore_index=self.config.semantic_loss_ignore_index)
            if auxiliary_logits is not None:
                aux = F.cross_entropy(
                    auxiliary_logits, labels, ignore_index=self.config.semantic_loss_ignore_index
                )
                loss = loss + self.config.auxiliary_loss_weight * aux

        return SemanticSegmenterOutput(loss=loss, logits=logits)


config = DINOv3ForSemanticSegmentationConfig(
    dinov3_config=dinvo3_config,
    backbone="facebook/dinov3-vitb16-pretrain-lvd1689m",
)
model = DINOv3ForSemanticSegmentation(config)
model


In [ ]:
load_

In [ ]:
"""Frozen DINOv3 ViT backbone + BEiT FPN neck + BEiT UperNet/FCN heads."""
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

from transformers import BeitConfig
from transformers.modeling_outputs import SemanticSegmenterOutput
from transformers.models.beit.modeling_beit import BeitFCNHead, BeitFPNNeck, BeitUperHead
from transformers.models.dinov3_vit.modeling_dinov3_vit import DINOv3ViTBackbone


def bn_to_gn(module: nn.Module, groups: int = 32) -> nn.Module:
    """Swap BatchNorm2d for GroupNorm: training batches mix labeled and unlabeled frames."""
    for name, child in module.named_children():
        if isinstance(child, nn.BatchNorm2d):
            setattr(module, name, nn.GroupNorm(groups, child.num_features))
        else:
            bn_to_gn(child, groups)
    return module


@dataclass
class DINOv3ForSemanticSegmentationConfig:
    checkpoint: str = "facebook/dinov3-vits16-pretrain-lvd1689m"
    num_labels: int = 19
    # Stage index k is the output of block k (stage 0 is the patch embedding).
    out_indices: tuple[int, ...] = (4, 6, 8, 12)
    use_auxiliary_head: bool = True
    auxiliary_loss_weight: float = 0.4
    ignore_index: int = 255
    pool_scales: tuple[int, ...] = (1, 2, 3, 6)
    auxiliary_channels: int = 256
    auxiliary_num_convs: int = 1
    auxiliary_concat_input: bool = False


class DINOv3ForSemanticSegmentation(nn.Module):
    def __init__(self, config: DINOv3ForSemanticSegmentationConfig) -> None:
        super().__init__()
        if len(config.out_indices) != 4:
            raise ValueError("out_indices must list 4 stage indices, e.g. (4, 6, 8, 12) for ViT-S/B.")
        self.config = config

        # Pretrained weights, frozen. DINOv3ViTBackbone applies the final LayerNorm,
        # drops the CLS and register tokens, and returns spatial feature maps.
        self.backbone = DINOv3ViTBackbone.from_pretrained(
            config.checkpoint,
            out_indices=list(config.out_indices),
        )
        self.backbone.requires_grad_(False).eval()

        head_cfg = BeitConfig(
            hidden_size=self.backbone.config.hidden_size,
            num_labels=config.num_labels,
            pool_scales=config.pool_scales,
            auxiliary_channels=config.auxiliary_channels,
            auxiliary_num_convs=config.auxiliary_num_convs,
            auxiliary_concat_input=config.auxiliary_concat_input,
        )
        self.fpn = bn_to_gn(BeitFPNNeck(head_cfg))
        self.decode_head = bn_to_gn(BeitUperHead(head_cfg))
        self.auxiliary_head = bn_to_gn(BeitFCNHead(head_cfg)) if config.use_auxiliary_head else None

    def train(self, mode: bool = True):
        super().train(mode)
        self.backbone.eval()  # frozen backbone always in eval mode
        return self

    def forward(self, pixel_values: torch.Tensor, labels: torch.Tensor | None = None) -> SemanticSegmenterOutput:
        _, _, height, width = pixel_values.shape

        with torch.no_grad():
            feature_maps = self.backbone(pixel_values).feature_maps
        feature_maps = self.fpn(tuple(feature.float() for feature in feature_maps))  # 1/4, 1/8, 1/16, 1/32

        up = lambda t: F.interpolate(t, size=(height, width), mode="bilinear", align_corners=False)
        logits = up(self.decode_head(feature_maps))
        auxiliary_logits = None
        if self.auxiliary_head is not None and self.training:
            auxiliary_logits = up(self.auxiliary_head(feature_maps))

        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits, labels, ignore_index=self.config.ignore_index)
            if auxiliary_logits is not None:
                aux = F.cross_entropy(auxiliary_logits, labels, ignore_index=self.config.ignore_index)
                loss = loss + self.config.auxiliary_loss_weight * aux

        return SemanticSegmenterOutput(loss=loss, logits=logits)


config = DINOv3ForSemanticSegmentationConfig()
model = DINOv3ForSemanticSegmentation(config)
model